# IOAI — 2024 First Stage Adversarial Attacks (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/trained_model.pth'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-adversarial-attacks/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 적대적 공격 (Adversarial Attacks, 베이스라인)

폴란드 AI 올림피아드 I · 2024 · 1단계. 사전학습된 숫자 분류기(`Net`, 28×28)를 **속이는** 미세한 변형을
만든다. 이미지를 사람 눈엔 비슷하게 유지(SSIM↑)하면서 분류 정확도를 떨어뜨린다.

**규칙**: 정규화([-1,1]) 원본과 변형 이미지의 **픽셀별 최대 차이 ≤ 0.3**. **채점**:
`criterion = 평균SSIM × (원본정확도 − 공격후정확도)` (정확도는 %). 점수: criterion<36 → 0, >42 → 100, 선형.

이 노트북은 **베이스라인** = `perturbe_dataset` 이 원본을 그대로 반환(공격 없음) → 정확도 변화 0 → **0점**.
모범답안(PGD 공격)을 참고해 `perturbe_dataset` 을 구현하라.

**제출**: `submission.npz` — `perturbed(N,28,28)` (공격된 val 이미지, [-1,1]).


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/trained_model.pth"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-adversarial-attacks/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"

# 속이려는 분류기 (고정)
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,1,0); self.conv2 = nn.Conv2d(32,64,3,1,0)
        self.pool = nn.MaxPool2d(3,2); self.fc1 = nn.Linear(64*11*11,128); self.fc2 = nn.Linear(128,10)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = F.relu(self.pool(self.conv2(x))); x = torch.flatten(x,1)
        return F.log_softmax(self.fc2(self.fc1(x)), 1)

def normalize_samples(samples):   # 이미지별 [-1,1] 정규화 (원문제와 동일)
    s = samples.reshape(-1, 784).astype("float64")
    mn = s.min(1, keepdims=True); mx = s.max(1, keepdims=True)
    return ((2*(s-mn)/(mx-mn)) - 1).reshape(-1, 28, 28)

net = Net().to(device)
net.load_state_dict(torch.load("data/trained_model.pth", map_location=device)); net.eval()
X_validation = normalize_samples(np.load("data/contest_validation_samples.npy")/255.).astype("float32")
y_validation = np.load("data/contest_validation_labels.npy")
print("val", X_validation.shape, "| device", device)


In [ ]:
def perturbe_dataset(original_dataset):
    """베이스라인: 공격 없이 원본을 그대로 반환 (정확도 변화 0 → 0점). TODO: 실제 공격 구현."""
    assert len(original_dataset.shape) == 3
    return original_dataset.copy()


In [ ]:
# val 공격 -> submission.npz
perturbed = perturbe_dataset(X_validation)
assert perturbed.shape == X_validation.shape
assert np.max(np.abs(perturbed - X_validation)) <= 0.3 + 1e-6, "L∞ 제약(0.3) 위반"
np.savez_compressed("submission.npz", perturbed=perturbed.astype("float32"))
print("submission.npz 저장:", perturbed.shape, "| L∞", round(float(np.max(np.abs(perturbed-X_validation))),3))


### 다음 단계
원본 유지로는 정확도 하락 0 → 0점. `perturbe_dataset` 을 **PGD**(분류기 손실을 키우는 방향으로 L∞≤0.3 이동)
로 구현하면 정확도가 92→0%로 떨어져 criterion≈48 → 100점. 모범답안 참고.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)